In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
import pickle

In [2]:
data = pd.read_csv(r'D:\Krish_Naik\Python\Deep Learning\Regression\Churn_Modelling.csv')
data.head()
                   

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocessing the data
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1) 

In [4]:
## Encoding the categorical data
labelencoder = LabelEncoder()
data['Gender'] = labelencoder.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [6]:
### One hot encoding the 'Geography' column
onehotencoder = OneHotEncoder(handle_unknown='ignore')
geography = onehotencoder.fit_transform(data['Geography'].values.reshape(-1,1)).toarray()
geography_df = pd.DataFrame(geography, columns=onehotencoder.get_feature_names_out(['Geography']))
geography_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [8]:
data =pd.concat([data.drop('Geography',axis=1),geography_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
#### Splitting the dataset into the Training set and Test set
X =data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [10]:
### Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
### Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
### Save the encoders and scaler for future use
with open('labelencoder.pkl', 'wb') as file:
    pickle.dump(labelencoder, file)

with open('onehotencoder.pkl', 'wb') as file:
    pickle.dump(onehotencoder, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


In [14]:
### ANN Regression Model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [15]:
model = Sequential(
    [
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1)  # Output layer for regression
])

## Compile the model
model.compile(optimizer='adam', loss='mean_absolute_error',metrics=['mae'])
model.summary()


d:\Krish_Naik\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
### Train the model
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Set up TensorBoard callback
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [17]:
## Set up EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [18]:
#Train the model
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test), 
                    validation_split=0.2, epochs=100,
                    batch_size=32,
                    callbacks=[early_stopping, tensorboard_callback]
                    )

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100377.6797 - mae: 100377.6797 - val_loss: 98513.1172 - val_mae: 98513.1172
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 99584.4062 - mae: 99584.4062 - val_loss: 96898.5703 - val_mae: 96898.5703
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 96745.8125 - mae: 96745.8125 - val_loss: 92755.6562 - val_mae: 92755.6562
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 91175.8438 - mae: 91175.8438 - val_loss: 85871.4375 - val_mae: 85871.4375
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 83118.7422 - mae: 83118.7422 - val_loss: 77116.9219 - val_mae: 77116.9219
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 73858.5078 - mae: 73858.5078 - val_loss: 68186.6094 - val_mae: 68186.6094
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 65031.6289 - mae: 65031.6289 - val_loss: 60600.5234 - val_mae: 60600.5234
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 

In [19]:
### Load extention tensorboard
%load_ext tensorboard

In [21]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 10656), started 0:00:18 ago. (Use '!kill 10656' to kill it.)

In [22]:
### Evaluate the model
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 50359.7891 - mae: 50359.7891
Test Loss: 50359.7890625, Test MAE: 50359.7890625


In [23]:
model.save('regression_model.h5')